# Day 13: Hallucination Prevention, Walked Through

Grounding an answer in retrieved documents helps a lot, but doesn't make
hallucination impossible. This notebook detects unsupported claims, verifies them
against sources, expresses calibrated uncertainty, handles contradictions, enforces
grounding, and wires it all into a full fact-checking pipeline.

Run each cell in order. Plain Python throughout, self-contained.

## Cell 1: What are hallucinations

A hallucination is a confidently stated claim that isn't actually supported by the
sources -- or isn't true at all. Wrong information stated with confidence is often
worse than no information: a user who gets "I don't know" looks elsewhere; a user
who gets a confident wrong answer may act on it directly.

## Cell 2: Detect hallucinations

Check each CLAIM independently, not the whole answer at once -- a whole-answer
average can hide one bad sentence among several good ones.

In [ ]:
import re

STOPWORDS = {"what", "is", "a", "an", "the", "are", "of", "in", "to", "and", "for", "it", "its"}

def keywords(text):
    return {w.strip("?.,!").lower() for w in text.split()} - STOPWORDS

def split_into_claims(answer):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', answer.strip()) if s.strip()]

def check_claim_support(claim, source_texts, threshold=0.5):
    claim_words = keywords(claim)
    if not claim_words:
        return 1.0, True
    source_words = set()
    for t in source_texts:
        source_words |= keywords(t)
    score = len(claim_words & source_words) / len(claim_words)
    return score, score >= threshold

sources = [
    "RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents before generating an answer.",
]
answer = (
    "RAG stands for Retrieval-Augmented Generation. "
    "It retrieves relevant documents before generating an answer. "
    "RAG was invented in 2019 by a team of twelve researchers at a major robotics lab."
)

for claim in split_into_claims(answer):
    score, supported = check_claim_support(claim, sources)
    status = "SUPPORTED" if supported else "UNSUPPORTED"
    print(f"[{status}] ({score:.2f}) {claim}")

## Cell 3: Extract and verify claims

Not every sentence is a checkable claim -- questions, opinions, and vague filler
should be filtered before verification, and a claim should be traced to its
SPECIFIC supporting document.

In [ ]:
OPINION_MARKERS = {"best", "worst", "great", "arguably", "should"}
GENERIC_WORDS = {"this", "that", "matters", "important", "lot"}

def classify_sentence(sentence):
    if sentence.strip().endswith("?"):
        return "question"
    words = set(w.strip("?.,!").lower() for w in sentence.split())
    if words & OPINION_MARKERS:
        return "opinion"
    kw = keywords(sentence)
    if len(sentence.split()) <= 3 or len(kw - GENERIC_WORDS) == 0:
        return "too_vague"
    return "factual"

def verify_claim(claim, documents):
    scores = [(doc["title"], check_claim_support(claim, [doc["text"]])[0]) for doc in documents]
    scores.sort(key=lambda pair: pair[1], reverse=True)
    return scores

documents = [
    {"title": "RAG Overview", "text": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents before generating an answer."},
    {"title": "RAG History", "text": "RAG was introduced by researchers combining retrieval systems with sequence-to-sequence generation models."},
]

example = "RAG stands for Retrieval-Augmented Generation. Is that the same as search? This matters a lot. RAG was introduced by researchers."
for sentence in split_into_claims(example):
    claim_type = classify_sentence(sentence)
    print(f"[{claim_type:9s}] {sentence}")
    if claim_type == "factual":
        print(f"             best source: {verify_claim(sentence, documents)[0]}")

## Cell 4: Express uncertainty

Calibrate phrasing to match actual confidence -- firm when certain, hedged when
unsure, honest "I don't know" when there's nothing to go on.

In [ ]:
def confidence_band(score):
    if score >= 0.8: return "high"
    if score >= 0.5: return "medium"
    if score >= 0.2: return "low"
    return "very_low"

def express_with_calibration(claim, score):
    band = confidence_band(score)
    if band == "high":
        return claim
    if band == "medium":
        return f"{claim.rstrip('.')}, though this is based on limited information."
    if band == "low":
        core = claim.rstrip(".")
        first, _, rest = core.partition(" ")
        first = first if first.isupper() else first[:1].lower() + first[1:]
        return f"This isn't certain, but it's possible that {first} {rest}."
    return "I don't have enough grounded information to answer that confidently."

test_claims = [
    ("RAG retrieves relevant documents before generating an answer.", 1.0),
    ("RAG researchers introduced documents.", 0.75),
    ("RAG models were trained using quantum computing hardware.", 0.25),
    ("Pineapple pizza originated in medieval France.", 0.0),
]
for claim, score in test_claims:
    print(f"[{confidence_band(score):8s}] {express_with_calibration(claim, score)}")

## Cell 5: Handle contradictions

When sources disagree on a number, surface both viewpoints instead of silently
picking one.

In [ ]:
def extract_numeric_facts(text, window=2):
    facts = []
    words = text.split()
    for i, word in enumerate(words):
        match = re.match(r'[\$]?(\d[\d.,]*)', word)
        if match:
            start, end = max(0, i - window), min(len(words), i + window + 1)
            context = set(w.strip("?.,!$%").lower() for w in words[start:i] + words[i+1:end])
            facts.append({"number": match.group(1), "context": context})
    return facts

def find_contradiction(doc_a, doc_b, threshold=2):
    facts_a, facts_b = extract_numeric_facts(doc_a["text"]), extract_numeric_facts(doc_b["text"])
    best = None
    for fa in facts_a:
        for fb in facts_b:
            if fa["number"] == fb["number"]:
                continue
            overlap = fa["context"] & fb["context"]
            if len(overlap) >= threshold:
                best = (fa["number"], fb["number"], overlap)
    return best

doc_a = {"title": "Pricing (Jan)", "text": "The premium plan costs $10 per month and includes 5 projects."}
doc_b = {"title": "Pricing (updated)", "text": "The premium plan costs $15 per month and includes 5 projects."}

conflict = find_contradiction(doc_a, doc_b)
if conflict:
    num_a, num_b, context = conflict
    print(f"Sources disagree (shared context: {context}):")
    print(f"  {doc_a['title']}: {num_a}")
    print(f"  {doc_b['title']}: {num_b}")
    print("  Can't resolve from text alone -- surfacing both rather than picking one.")

## Cell 6: Enforce grounding

Strict enforcement actively removes unsupported claims, falling back to an honest
"I don't know" if nothing is grounded at all.

In [ ]:
def enforce_strict(answer, source_texts, threshold=0.5):
    kept, removed = [], []
    factual_count = 0
    factual_failed = 0
    for sentence in split_into_claims(answer):
        if classify_sentence(sentence) != "factual":
            kept.append(sentence)
            continue
        factual_count += 1
        score, supported = check_claim_support(sentence, source_texts, threshold)
        if supported:
            kept.append(sentence)
        else:
            removed.append(sentence)
            factual_failed += 1
    if factual_count and factual_failed == factual_count:
        return "I don't have enough grounded information in the provided sources to answer that confidently.", removed
    return " ".join(kept), removed

mixed_answer = (
    "RAG stands for Retrieval-Augmented Generation. "
    "It retrieves relevant documents before generating an answer. "
    "RAG was invented in 2019 by a team of twelve researchers at a major robotics lab."
)
final, removed = enforce_strict(mixed_answer, sources)
print(f"Final: {final}")
print(f"Removed: {removed}")

## Cell 7: Fact-checking pipeline

The full loop: extract claims, verify each, score overall factuality, revise.

In [ ]:
def run_pipeline(answer, source_texts):
    claims = split_into_claims(answer)
    scored = []
    for c in claims:
        claim_type = classify_sentence(c)
        if claim_type != "factual":
            scored.append((c, claim_type, None))
            continue
        score, _ = check_claim_support(c, source_texts)
        scored.append((c, "factual", score))

    factual_scores = [s for _, t, s in scored if t == "factual"]
    avg = sum(factual_scores) / len(factual_scores) if factual_scores else 1.0

    revised_parts = []
    for c, t, s in scored:
        if t != "factual" or s >= 0.5:
            revised_parts.append(c)
        else:
            revised_parts.append(express_with_calibration(c.rstrip("."), s))
    return avg, " ".join(revised_parts)

test_answer = (
    "RAG stands for Retrieval-Augmented Generation. "
    "It retrieves relevant documents before generating an answer. "
    "RAG was introduced by researchers combining retrieval and generation. "
    "RAG won a major international award in 2019."
)
all_sources = [documents[0]["text"], documents[1]["text"]]
avg_score, revised = run_pipeline(test_answer, all_sources)
print(f"Average factuality: {avg_score:.2f}")
print(f"Revised: {revised}")

## Cell 8: Integration example

Measuring that hallucination prevention improves factual consistency without
hurting relevance.

In [ ]:
def avg_factual_claim_confidence(answer, source_texts):
    """
    IMPORTANT: whole-answer word-overlap (like factual_consistency() in
    Day 7) is the WRONG metric for this comparison -- it was tried first
    here and it actually went DOWN after fact-checking, which looked
    like fact-checking made things worse. The real reason: the honest
    "I don't have enough grounded information..." fallback message is
    itself built from words that aren't in any source ("confidently",
    "grounded", "enough") -- so replacing a fabricated claim with an
    honest admission can make a crude whole-answer word-overlap score
    go DOWN, even though the answer got MORE trustworthy, not less.
    The correct comparison is the average confidence of the FACTUAL
    CLAIMS that remain assertions (ignoring boilerplate honesty
    statements, which were never meant to be "grounded" themselves) --
    this is exactly what fact_checking_pipeline.py's
    score_overall_factuality() actually measures.
    """
    claims = split_into_claims(answer)
    scores = []
    for c in claims:
        if classify_sentence(c) != "factual":
            continue
        if c.startswith("I don't have enough grounded information"):
            continue  # an honesty fallback isn't itself a claim to score
        score, _ = check_claim_support(c, source_texts)
        scores.append(score)
    return sum(scores) / len(scores) if scores else 1.0

original_answer = test_answer
_, revised_answer = run_pipeline(original_answer, all_sources)

print(f"Original avg factual claim confidence: {avg_factual_claim_confidence(original_answer, all_sources):.2f}")
print(f"Revised avg factual claim confidence:  {avg_factual_claim_confidence(revised_answer, all_sources):.2f}")
print(f"\nOriginal: {original_answer}")
print(f"Revised:  {revised_answer}")

## Cell 9: Key takeaways

- Hallucinations are a real, documented failure mode, even in RAG systems designed
  specifically to reduce them -- grounding helps, but doesn't guarantee accuracy.
- Check claims individually, not the whole answer as one block -- a whole-answer
  average can hide one fabricated sentence among several good ones.
- Not every sentence is a checkable claim -- filter out questions, opinions, and
  vague filler before sending anything to verification.
- Trace a claim to its SPECIFIC supporting document, not just "grounded somewhere"
  -- this matters for citation and for spotting when sources disagree.
- Calibrate confidence expression to the actual evidence: state firmly, hedge, or
  admit "I don't know" -- a system that sounds equally confident regardless of
  grounding is actively misleading.
- When sources genuinely disagree, surface both viewpoints rather than silently
  picking one.
- Strict grounding enforcement can remove unsupported claims and still preserve
  relevance -- hallucination prevention is a correction, not a quality tradeoff.
- Always measure the before/after with real quality metrics, not just intuition --
  this project has repeatedly found that a technique "should" show an effect but
  initially didn't, until the measurement itself was checked carefully.